# C2 — Transient synchrony: trajectories, decorrelation, and readout

Companion to the flagship unit
[`part2-case-studies/C2`](../part2-case-studies/C2-transient-synchrony.md).

A PN–LN antennal lobe model that reproduces the two signatures the locust literature
reports:

1. **progressive decorrelation** — odour representations start correlated and separate
   over the transient;
2. **the transient carries the code** — classification accuracy peaks *during* the
   transient and falls at the fixed point (Mazor & Laurent 2005).

Then the part that matters for the course: a **spectral-radius-matched random network
null**. If a generic recurrent network does the same thing, you have discovered a
property of recurrent dynamics, not of the antennal lobe. See
[S-07](../structures/S-07-random-matrices-and-chaos.md).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

NP, NL, dt = 60, 20, 1.0        # PNs, LNs, timestep (ms)
tP, tL, tA = 20., 50., 400.     # PN, GABA_slow, adaptation time constants (ms)
relu = lambda z: np.maximum(z, 0.)
rng  = np.random.default_rng(0)
spw  = lambda a, b, g, p=.5: (rng.random((a,b)) < p)*rng.gamma(3., 1/3., (a,b))*g/(b*p)

WLP, WPL = spw(NL, NP, 3.0), spw(NP, NL, 12.0)    # PN->LN excitation, LN-|PN inhibition
print(f'PN->LN {WLP.shape}, LN->PN {WPL.shape}')

## 1. The model

$$\tau_P \dot P = -P + [\,s - W_{PL}L - \alpha A\,]_+,\qquad
\tau_L \dot L = -L + [\,W_{LP}P\,]_+,\qquad
\tau_A \dot A = -A + P.$$

Slow GABA and slower adaptation are what make the transient long relative to the PN
membrane time constant. The odour bank is deliberately *overlapping* — 14 of 18 activated
glomeruli shared between odours — so decorrelation has real work to do.

In [ ]:
def odor_bank(n=8, k=18, shared=14, seed=3):
    r = np.random.default_rng(seed); S = np.zeros((n, NP))
    base = r.choice(NP, shared, replace=False)
    rest = np.setdiff1d(np.arange(NP), base)
    for i in range(n):
        idx = np.concatenate([base, r.choice(rest, k-shared, replace=False)])
        S[i, idx] = r.gamma(3., 1/3., k)*2.0
    return S

def run(s, T=2000, on=200, off=1400, alpha=3.0):
    P = np.zeros(NP); L = np.zeros(NL); A = np.zeros(NP); R = np.zeros((T, NP))
    for t in range(T):
        d = s if on < t < off else 0.
        P += dt*(-P + relu(d - WPL@L - alpha*A))/tP
        L += dt*(-L + relu(WLP@P))/tL
        A += dt*(-A + P)/tA
        R[t] = P
    return R

S = odor_bank()
Mact = np.array([run(s) for s in S])       # (8 odours, 2000 ms, 60 PNs)
print('activity tensor', Mact.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, i in zip(axes, [0, 1]):
    im = ax.imshow(Mact[i].T, aspect='auto', origin='lower', cmap='magma',
                   extent=[0, 2000, 0, NP])
    ax.axvline(200, color='w', ls='--', lw=1); ax.axvline(1400, color='w', ls='--', lw=1)
    ax.set_xlabel('time (ms)'); ax.set_ylabel('PN'); ax.set_title(f'odour {i}')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## 2. Decorrelation and the accuracy peak

For each time point: mean pairwise correlation between odour representations, and 8-way
nearest-centroid classification accuracy on noisy single trials.

In [ ]:
cc = lambda a, b: float((a-a.mean())@(b-b.mean()) /
                        (np.linalg.norm(a-a.mean())*np.linalg.norm(b-b.mean()) + 1e-12))
pairs = [(i, j) for i in range(8) for j in range(i)]
noise, R_ = 0.6, np.random.default_rng(7)

def profile(Mt):
    ts = np.arange(205, 1460, 15)
    corr, acc, norm = [], [], []
    for t in ts:
        C = Mt[:, t]
        X = C[:, None, :] + noise*R_.standard_normal((8, 120, C.shape[1]))
        pred = np.argmin(((X[:, :, None, :] - C[None, None])**2).sum(-1), -1)
        corr.append(np.mean([cc(C[i], C[j]) for i, j in pairs]))
        acc.append((pred == np.arange(8)[:, None]).mean())
        norm.append(np.linalg.norm(C, axis=1).mean())
    return ts, np.array(corr), np.array(acc), np.array(norm)

ts, corr, acc, norm = profile(Mact)
print(f'input correlation      {np.mean([cc(S[i], S[j]) for i, j in pairs]):.3f}')
print(f'PN correlation, early  {corr[:3].mean():.3f}')
print(f'PN correlation, late   {corr[-20:].mean():.3f}')
print(f'peak accuracy {acc.max():.2f} at t={ts[acc.argmax()]} ms; '
      f'fixed-point accuracy {acc[-20:].mean():.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ts, corr, lw=2, label='mean pairwise correlation')
ax.plot(ts, acc, lw=2, label='8-way accuracy')
ax.plot(ts, norm/norm.max(), lw=1.5, ls=':', color='grey', label='|r| (normalised)')
ax.axvspan(200, 500, color='orange', alpha=.12, label='transient')
ax.set_xlabel('time (ms)'); ax.set_ylabel('value')
ax.set_title('Accuracy peaks in the transient, not at the fixed point')
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

This is the Mazor–Laurent result: the *transient* is where the odours are most
discriminable. A code read out at the fixed point throws away most of the separation.

## 3. Trajectory geometry

Project all eight trajectories onto the leading PCs of the pooled activity.

In [ ]:
Z = Mact[:, 200:1400].reshape(-1, NP); mu = Z.mean(0)
U, sv, Vt = np.linalg.svd(Z - mu, full_matrices=False)
ev = sv**2
print('PC variance fractions:', np.round((ev/ev.sum())[:5], 3))
print(f'participation ratio: {ev.sum()**2/(ev**2).sum():.1f}')

fig, ax = plt.subplots(figsize=(5.5, 5))
for i in range(8):
    traj = (Mact[i, 200:1400] - mu) @ Vt[:2].T
    ax.plot(traj[:, 0], traj[:, 1], lw=1.4, alpha=.85)
    ax.plot(*traj[0], 'o', ms=5, color='k')
    ax.plot(*traj[-1], 's', ms=5, color='crimson')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Odour trajectories (circle = onset, square = fixed point)')
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 4. The null that matters

Now the discipline from [S-07](../structures/S-07-random-matrices-and-chaos.md). Replace
the structured PN–LN circuit with a **random** recurrent network matched on spectral
radius, driven by the same odours. A random network is strongly non-normal, so it also
produces long amplified transients.

Predict before you run it: does the random network decorrelate the odours too? If it
does, decorrelation is a generic property of recurrent dynamics and tells you nothing
specific about the antennal lobe.

In [ ]:
def spectral_radius(W): return np.abs(np.linalg.eigvals(W)).max()

# A reservoir null: dense random recurrence, strongly non-normal, tuned just below the
# g=1 transition (S-07) where memory and transient amplification are maximal. This is the
# most favourable setting for the reservoir hypothesis, so it is the fair null.
rho_null = 0.95
Wr = rng.standard_normal((NP, NP))
Wr *= rho_null/spectral_radius(Wr)
nonnormality = (np.linalg.norm(Wr.T@Wr - Wr@Wr.T, 'fro') /
                np.linalg.norm(Wr, 'fro')**2)
print(f'null spectral radius {spectral_radius(Wr):.3f}, '
      f'non-normality index {nonnormality:.3f}')

def run_null(s, T=2000, on=200, off=1400, alpha=3.0):
    P = np.zeros(NP); A = np.zeros(NP); R = np.zeros((T, NP))
    for t in range(T):
        d = s if on < t < off else 0.
        P += dt*(-P + relu(d + Wr@P - alpha*A))/tP
        A += dt*(-A + P)/tA
        R[t] = P
    return R

Mnull = np.array([run_null(s) for s in S])
ts_n, corr_n, acc_n, _ = profile(Mnull)
print(f'null: correlation {corr_n[:3].mean():.3f} -> {corr_n[-20:].mean():.3f}, '
      f'peak accuracy {acc_n.max():.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ts, corr, lw=2, label='PN-LN circuit')
axes[0].plot(ts_n, corr_n, lw=2, ls='--', label='matched random null')
axes[0].set_ylabel('mean pairwise correlation')
axes[1].plot(ts, acc, lw=2, label='PN-LN circuit')
axes[1].plot(ts_n, acc_n, lw=2, ls='--', label='matched random null')
axes[1].set_ylabel('8-way accuracy')
for a in axes:
    a.set_xlabel('time (ms)'); a.legend(); a.grid(alpha=.3)
plt.suptitle('Is the decorrelation specific, or generic?')
plt.tight_layout(); plt.show()

**The null does not decorrelate.** It sits at roughly the input correlation (~0.5) for the
whole trial, while the structured PN–LN circuit drives it down to ~0.16. So decorrelation
here is *not* a generic consequence of recurrent dynamics at the edge of chaos — it
requires the structured, delayed, adapting inhibition. That is a real claim, and this is
the shape of the evidence it needs.

Note that the null still classifies well — often *better* early on, because it preserves
input geometry instead of reshaping it. Decorrelation and discriminability are different
objectives, and conflating them is one of the commonest errors in this literature.

## Things to try

1. **The slope test.** Build a parametric family of odour pairs spanning a range of
   initial correlations, and regress asymptotic on initial correlation for circuit and
   null. Slope $\approx 0$ means genuine whitening; $\approx 1$ means the ordering is
   merely preserved and rescaled. This is the single most informative experiment in the
   notebook, and it is what turns "it decorrelates" into a falsifiable statement.
2. **Match non-normality too.** The null above matches only the spectral radius. Match
   $\|W^\top W - WW^\top\|_F/\|W\|_F^2$ as well and see how much of the gap survives.
3. **Build the heteroclinic competitor.** Implement the generalised Lotka–Volterra
   winnerless-competition model from C2 §2.3 and check its signature: dwell times growing
   like $\lambda_u^{-1}\ln(1/\varepsilon)$, i.e. *linearly in log noise*. Neither the
   circuit above nor the null does that — it is the sharpest discriminator available.
4. **Topology.** Run persistent homology on the trajectory bundles
   ([S-05 Ex 5.3](../structures/S-05-toroidal-topology-of-grid-cells.md)). Note in advance
   that it *cannot* separate open-chain heteroclinic from non-normal amplification — both
   are contractible. Knowing what your instrument cannot do is half of using it.